**VIRTUAL MOUSE CONTROLLER**

- **INDEX FINGER TO MOVE CURSOR.**
- **PINCH TO CLICK.**


In [6]:
import cv2
import mediapipe as mp
import time
import math
import pyautogui

wScr, hScr = pyautogui.size()
cap = cv2.VideoCapture(0)

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

p_time = 0

while True:
    success, img = cap.read()
    if not success:
        break

    img = cv2.flip(img, 1)
    hCam, wCam, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            x1 = int(hand_landmarks.landmark[8].x * wCam)
            y1 = int(hand_landmarks.landmark[8].y * hCam)
            x2 = int(hand_landmarks.landmark[4].x * wCam)
            y2 = int(hand_landmarks.landmark[4].y * hCam)

            # Map camera coords to screen
            screen_x = int(x1 * wScr / wCam)
            screen_y = int(y1 * hScr / hCam)
            pyautogui.moveTo(screen_x, screen_y)

            # Pinch gesture for click
            dist = math.hypot(x2 - x1, y2 - y1)
            if dist < 30:
                pyautogui.click()
                cv2.circle(img, (x1, y1), 12, (0, 0, 255), -1)
                time.sleep(0.2)

            cv2.circle(img, (x1, y1), 10, (255, 0, 255), -1)

    c_time = time.time()
    fps = 1 / (c_time - p_time)
    p_time = c_time
    cv2.putText(img, f"FPS: {int(fps)}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Virtual Mouse", img)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()